---
title: 'Discretizing the Mean-Reversion Strategy: EWM Fair Value, Return Autocovariance, and Gross PnL'
date: 2026-04-24
description: "A discrete OU model turns an EWM fair-value estimate into a return filter and yields closed-form gross PnL formulas."
categories: [Quantitative Finance, Trading Strategies, Stochastic Processes]
content-type: "series"
series-id: "mean-reversion-strategy"
series-title: "Expected Performance of a Mean-Reversion Trading Strategy"
series-part: 5
project-ids: [mean-reversion-strategy]
---

In [Part 1](../mean-reversion-strategy-1/index.qmd) we derived the continuous-time Sharpe ratio of a mean-reversion strategy when the trader knows the Ornstein-Uhlenbeck model exactly. [Part 2](../mean-reversion-strategy-2/) showed how a constant fair-value bias raises risk, [Part 3](../mean-reversion-strategy-3/) showed how trailing estimators create a persistent covariance penalty, and [Part 4](../mean-reversion-strategy-4/) linked those bias channels to finite-sample parameter estimation.

Those results are useful as limiting calculations, but they do not yet describe the object that a trader actually implements. Positions are rebalanced at discrete times, fair value is usually estimated from sampled prices, and proportional transaction costs make continuous rebalancing a poor modeling abstraction. This post therefore restarts the strategy in discrete time. The goal is not to abandon the OU model, but to express it in the form needed for turnover, transaction costs, and finite-horizon PnL distributions.

The central observation is that an exponentially weighted moving-average fair-value estimate turns the mean-reversion signal into a linear filter of past returns. Under a Gaussian OU model, returns are jointly Gaussian, so a linear signal makes cumulative PnL a quadratic form in a Gaussian vector. Part 5 derives the discrete OU return autocovariance, the EWM signal identity, the no-lookahead trading rule, and the stationary one-period gross PnL formulas. Part 6 will use the same objects to build the full quadratic-form distribution.


In [ ]:
#| echo: false
#| output: false
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

rng = np.random.default_rng(42)


def simulate_ou_ar1(n_paths, n_steps, a, omega, rng):
    """Simulate stationary AR(1) OU mispricing X_i = a X_{i-1} + eps_i."""
    eps_std = omega * np.sqrt(1 - a**2)
    X = np.empty((n_paths, n_steps + 1))
    X[:, 0] = rng.normal(0.0, omega, size=n_paths)
    for i in range(1, n_steps + 1):
        X[:, i] = a * X[:, i - 1] + rng.normal(0.0, eps_std, size=n_paths)
    return X


def ewm_fair_value(price, q):
    """Return EWM fair value and estimated mispricing z = price - vhat."""
    price = np.asarray(price, dtype=float)
    was_1d = price.ndim == 1
    if was_1d:
        price = price[None, :]
    vhat = np.empty_like(price)
    z = np.empty_like(price)
    vhat[:, 0] = price[:, 0]
    z[:, 0] = 0.0
    for i in range(1, price.shape[1]):
        vhat[:, i] = q * vhat[:, i - 1] + (1 - q) * price[:, i]
        z[:, i] = price[:, i] - vhat[:, i]
    if was_1d:
        return vhat[0], z[0]
    return vhat, z


def ewm_return_filter(returns, q):
    """Compute z_i = q(z_{i-1} + r_i) from returns r_i."""
    returns = np.asarray(returns, dtype=float)
    was_1d = returns.ndim == 1
    if was_1d:
        returns = returns[None, :]
    z = np.zeros((returns.shape[0], returns.shape[1] + 1))
    for i in range(1, returns.shape[1] + 1):
        z[:, i] = q * (z[:, i - 1] + returns[:, i - 1])
    if was_1d:
        return z[0]
    return z


def return_cov_theory(h, a, omega2):
    """Theoretical Cov(r_i, r_{i-h}) for integer h >= 1."""
    h = np.asarray(h, dtype=float)
    return -omega2 * (1 - a)**2 * a**(h - 1)


def return_var_theory(a, omega2):
    """Theoretical Var(r_i)."""
    return 2 * omega2 * (1 - a)


def return_autocorr_theory(h, a):
    """Theoretical Corr(r_i, r_{i-h}) for integer h >= 1."""
    h = np.asarray(h, dtype=float)
    return -0.5 * (1 - a) * a**(h - 1)


def empirical_return_autocorr(returns, max_lag):
    """Estimate return autocorrelation by pooling paths and time points."""
    returns = np.asarray(returns, dtype=float)
    out = []
    for h in range(1, max_lag + 1):
        x = returns[:, h:].reshape(-1)
        y = returns[:, :-h].reshape(-1)
        x = x - x.mean()
        y = y - y.mean()
        out.append(np.mean(x * y) / np.sqrt(np.mean(x**2) * np.mean(y**2)))
    return np.array(out)


def stationary_formulas(a, q, omega2, gamma=1.0):
    """Return stationary variance, covariance, PnL, Sharpe, and turnover terms."""
    var_r = 2 * omega2 * (1 - a)
    var_z = 2 * omega2 * q**2 * (1 - a) / ((1 + q) * (1 - a * q))
    cov_r_zprev = -omega2 * q * (1 - a)**2 / (1 - a * q)
    mean_pnl = -gamma * cov_r_zprev
    var_pnl = gamma**2 * (var_r * var_z + cov_r_zprev**2)
    local_sharpe = mean_pnl / np.sqrt(var_pnl)
    var_delta_z = (
        2 * q**2 * omega2 * (1 - a) * (3 - a - q - a * q)
        / ((1 + q) * (1 - a * q))
    )
    turnover_mean = gamma * np.sqrt(2 / np.pi) * np.sqrt(var_delta_z)
    return {
        "var_r": var_r,
        "var_z": var_z,
        "cov_r_zprev": cov_r_zprev,
        "mean_pnl": mean_pnl,
        "var_pnl": var_pnl,
        "local_sharpe": local_sharpe,
        "var_delta_z": var_delta_z,
        "turnover_mean": turnover_mean,
    }


def pnl_linear_mean_reversion(X, q, gamma=1.0):
    """Compute returns, EWM fair value, no-lookahead signal, and one-period PnL."""
    X = np.asarray(X, dtype=float)
    was_1d = X.ndim == 1
    if was_1d:
        X = X[None, :]
    returns = np.diff(X, axis=1)
    vhat, z = ewm_fair_value(X, q)
    signal = -gamma * z[:, :-1]
    pnl = signal * returns
    result = {"returns": returns, "vhat": vhat, "z": z, "signal": signal, "pnl": pnl}
    if was_1d:
        return {key: value[0] for key, value in result.items()}
    return result


# Discrete OU Model

We normalize the true fair value to zero in this post, so the log price equals the mispricing. This is a modeling normalization: it lets the algebra focus on the estimation and trading mechanism, not on an externally observed fair-value process. The continuous-time reference model is

$$
dX_t = -\theta X_t\,dt + \sigma\,dW_t
$$

For a fixed rebalancing interval $\Delta$, the exact discrete-time model is

$$
X_i = aX_{i-1} + \varepsilon_i,\qquad a=e^{-\theta\Delta}
$$

where

$$
\varepsilon_i\sim N(0,\sigma_\varepsilon^2),\qquad
\sigma_\varepsilon^2=\omega^2(1-a^2),\qquad
\omega^2=\operatorname{Var}(X_i)=\frac{\sigma^2}{2\theta}
$$

The price and one-period return are

$$
p_i=X_i,\qquad r_i=p_i-p_{i-1}=X_i-X_{i-1}
$$

Notation will remain discrete from this point forward. The parameter $a$ is the OU persistence over one rebalancing interval, $\omega^2$ is the stationary variance of the mispricing, $q$ will denote the persistence of the fair-value estimator, and $\gamma$ will denote the position scale. The signal will be defined using information available before the return it earns, which is the convention that prevents lookahead bias.


# Return Autocovariance

The first discrete-time object we need is the autocovariance of returns. The process $X_i$ is positively autocorrelated, but returns are negatively autocorrelated because a high mispricing tends to be followed by a correction.

**Proposition 5.1.** Under the stationary AR(1) model,

$$
\operatorname{Var}(r_i)=2\omega^2(1-a)
$$

and for $h\ge 1$,

$$
\operatorname{Cov}(r_i,r_{i-h})
=-\omega^2(1-a)^2a^{h-1}
$$

Stationarity gives

$$
\operatorname{Cov}(X_i,X_j)=\omega^2 a^{|i-j|}
$$

For the variance,

$$
\begin{aligned}
\operatorname{Var}(r_i)
&=\operatorname{Var}(X_i-X_{i-1}) \\
&=\operatorname{Var}(X_i)+\operatorname{Var}(X_{i-1})
-2\operatorname{Cov}(X_i,X_{i-1}) \\
&=2\omega^2-2\omega^2a \\
&=2\omega^2(1-a)
\end{aligned}
$$

For $h\ge 1$,

$$
r_i=X_i-X_{i-1},\qquad
r_{i-h}=X_{i-h}-X_{i-h-1}
$$

Therefore,

$$
\begin{aligned}
\operatorname{Cov}(r_i,r_{i-h})
&=\operatorname{Cov}(X_i-X_{i-1},X_{i-h}-X_{i-h-1}) \\
&=\operatorname{Cov}(X_i,X_{i-h})
-\operatorname{Cov}(X_i,X_{i-h-1}) \\
&\quad
-\operatorname{Cov}(X_{i-1},X_{i-h})
+\operatorname{Cov}(X_{i-1},X_{i-h-1}) \\
&=\omega^2a^h-\omega^2a^{h+1}
-\omega^2a^{h-1}+\omega^2a^h \\
&=\omega^2a^{h-1}(2a-a^2-1) \\
&=-\omega^2(1-a)^2a^{h-1}
\end{aligned}
$$

This negative autocovariance is the sampled signature of mean reversion. It is also the source of the positive expected gross PnL derived below: a position that leans against the recent price movement is, on average, aligned with the next correction.


In [ ]:
#| code-fold: true
#| code-summary: "Show simulation code"
#| label: fig-return-autocovariance
#| fig-cap: 'OU return autocorrelation. Simulation dots are compared with the theoretical correlation $-\frac{1}{2}(1-a)a^{h-1}$ for daily and monthly sampling of the same $\theta=1, \sigma=1$ OU model.'
theta = 1.0
sigma = 1.0
omega = sigma / np.sqrt(2 * theta)
max_lag = 30
lags = np.arange(1, max_lag + 1)
settings = [
    ("Daily sampling", 1 / 252, 5000, 756),
    ("Monthly sampling", 1 / 12, 5000, 360),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, (title, delta, n_paths, n_steps) in zip(axes, settings):
    a = np.exp(-theta * delta)
    X = simulate_ou_ar1(n_paths, n_steps, a, omega, np.random.default_rng(int(1_000_000 * delta)))
    returns = np.diff(X, axis=1)
    empirical = empirical_return_autocorr(returns, max_lag)
    theory = return_autocorr_theory(lags, a)
    ax.plot(lags, theory, color="black", lw=2, label="theory")
    ax.scatter(lags, empirical, s=28, color="steelblue", alpha=0.8, label="simulation")
    ax.axhline(0, color="black", lw=0.8, alpha=0.4)
    ax.set_title(f"{title}: $a={a:.4f}$")
    ax.set_xlabel("lag $h$")
    ax.set_ylabel("return autocorrelation")
    ax.legend()

fig.tight_layout()


# EWM Fair Value

The trader does not observe a separate fair-value process. Instead, the trader estimates fair value with an exponentially weighted moving average of price:

$$
\widehat v_i = q\widehat v_{i-1} + (1-q)p_i,\qquad 0\le q<1
$$

The estimated mispricing is

$$
\widetilde X_i = p_i-\widehat v_i
$$

Since $p_i=X_i$, the estimated mispricing obeys a simple recursion:

$$
\begin{aligned}
\widetilde X_i
&=p_i-\widehat v_i \\
&=p_i-q\widehat v_{i-1}-(1-q)p_i \\
&=q(p_i-\widehat v_{i-1}) \\
&=q(p_i-p_{i-1}+p_{i-1}-\widehat v_{i-1}) \\
&=q(r_i+\widetilde X_{i-1})
\end{aligned}
$$

With the initialization $\widetilde X_0=0$, iteration gives

$$
\widetilde X_i
=q\sum_{j=1}^{i}q^{i-j}r_j
$$

This identity is the key structural step. The fair-value estimate has converted the value signal into an exponentially weighted filter of past returns. A slow estimator, with $q$ close to one, retains a long memory of past returns. A fast estimator, with smaller $q$, tracks price more aggressively and leaves less estimated mispricing to trade against.


In [ ]:
#| code-fold: true
#| code-summary: "Show simulation code"
#| label: fig-ewm-signal-reconstruction
#| fig-cap: 'EWM estimated mispricing and its filtered-return reconstruction. The two series coincide because $Z_i=q(Z_{i-1}+r_i)$, up to numerical precision.'
theta = 1.0
sigma = 1.0
delta = 1 / 252
a = np.exp(-theta * delta)
omega = sigma / np.sqrt(2 * theta)
q = 0.97
n_steps = 252
X = simulate_ou_ar1(1, n_steps, a, omega, np.random.default_rng(7))[0]
returns = np.diff(X)
_, z_direct = ewm_fair_value(X, q)
z_filter = ewm_return_filter(returns, q)
residual = z_direct - z_filter

time = np.arange(n_steps + 1) * delta
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, height_ratios=[3, 1])
axes[0].plot(time, z_direct, label="$Z_i=p_i-\\widehat v_i$", color="steelblue", lw=2)
axes[0].plot(time, z_filter, "--", label="filtered returns", color="darkorange", lw=1.8)
axes[0].set_ylabel("estimated mispricing")
axes[0].legend()
axes[0].set_title(f"EWM reconstruction with $q={q}$")

axes[1].plot(time, residual, color="crimson", lw=1.5)
axes[1].axhline(0, color="black", lw=0.8, alpha=0.5)
axes[1].set_xlabel("time in years")
axes[1].set_ylabel("difference")
axes[1].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
fig.tight_layout()


# Signal Timing

The strategy trades against the estimated mispricing. To avoid lookahead bias, the position held during the return $r_i$ must be determined before $r_i$ is observed. Define

$$
s_i=-\gamma\widetilde X_{i-1}
$$

where $\gamma>0$ is a position scale. Substituting the EWM filter identity gives

$$
s_i
=-\gamma q\sum_{j=1}^{i-1}q^{i-1-j}r_j
$$

The negative sign is the value-trading sign: when the price is above the estimated fair value, the strategy is short; when the price is below the estimated fair value, the strategy is long. The lag is equally important. If $s_i$ were based on $\widetilde X_i$, the position would already contain the return $r_i$ it is supposed to earn.

This timing convention also reveals the connection to the next post. The signal is a linear transformation of lagged returns, so cumulative PnL is a quadratic expression in the Gaussian return vector. Part 6 will make that statement precise by constructing the EWM memory matrix and the symmetric PnL matrix.


# Gross PnL

The one-period gross PnL is

$$
\delta P_i = r_i s_i=-\gamma r_i\widetilde X_{i-1}
$$

In stationarity,

$$
\widetilde X_{i-1}
=q\sum_{h=1}^{\infty}q^{h-1}r_{i-h}
$$

Using Proposition 5.1,

$$
\begin{aligned}
E[\delta P_i]
&=-\gamma q\sum_{h=1}^{\infty}q^{h-1}
\operatorname{Cov}(r_i,r_{i-h}) \\
&=-\gamma q\sum_{h=1}^{\infty}q^{h-1}
\left[-\omega^2(1-a)^2a^{h-1}\right] \\
&=\gamma q\omega^2(1-a)^2
\sum_{h=1}^{\infty}(aq)^{h-1} \\
&=\gamma\frac{q\omega^2(1-a)^2}{1-aq}
\end{aligned}
$$

The expected gross PnL is positive because $r_i$ is negatively correlated with the return history entering the estimated mispricing. The EWM estimator aggregates those past returns, and the trading rule takes the opposite side of the estimated deviation.


In [ ]:
#| code-fold: true
#| code-summary: "Show plotting code"
#| label: fig-gross-pnl-vs-q
#| fig-cap: 'Expected one-period gross PnL as a function of EWM persistence $q$, with $\omega=1$ and $\gamma=1$. More persistent OU states generate different tradeoffs through the factor $q(1-a)^2/(1-aq)$.'
q_grid = np.linspace(0.001, 0.999, 400)
a_values = [0.60, 0.80, 0.95, 0.99]
omega2 = 1.0
gamma = 1.0

fig, ax = plt.subplots(figsize=(10, 5))
for a in a_values:
    mean_pnl = stationary_formulas(a, q_grid, omega2, gamma)["mean_pnl"]
    ax.plot(q_grid, mean_pnl, lw=2, label=f"$a={a}$")
ax.set_xlabel("EWM persistence $q$")
ax.set_ylabel("$E[\\delta P_i]$")
ax.set_title("Stationary expected one-period gross PnL")
ax.legend()
fig.tight_layout()


# Variance

For risk-adjusted performance we also need the covariance structure of $r_i$ and the lagged estimated mispricing. Define $Z_i=\widetilde X_i$. The joint state obeys

$$
X_i=aX_{i-1}+\varepsilon_i
$$

and

$$
\begin{aligned}
Z_i
&=qZ_{i-1}+q(X_i-X_{i-1}) \\
&=qZ_{i-1}+q(a-1)X_{i-1}+q\varepsilon_i
\end{aligned}
$$

Solving the stationary covariance equations gives

$$
\operatorname{Var}(X_i)=\omega^2
$$

$$
\operatorname{Cov}(X_i,Z_i)
=\frac{\omega^2q(1-a)}{1-aq}
$$

and

$$
\operatorname{Var}(Z_i)
=\frac{2\omega^2q^2(1-a)}{(1+q)(1-aq)}
$$

Because

$$
r_i=(a-1)X_{i-1}+\varepsilon_i
$$

and $\varepsilon_i$ is independent of $Z_{i-1}$,

$$
\operatorname{Cov}(r_i,Z_{i-1})
=-\frac{\omega^2q(1-a)^2}{1-aq}
$$

This recovers the mean formula in a compact form:

$$
E[\delta P_i]
=-\gamma\operatorname{Cov}(r_i,Z_{i-1})
$$

For centered jointly Gaussian variables $U$ and $V$,

$$
\operatorname{Var}(UV)
=\operatorname{Var}(U)\operatorname{Var}(V)
+\operatorname{Cov}(U,V)^2
$$

Applying this identity to $U=r_i$ and $V=Z_{i-1}$ gives

$$
\operatorname{Var}(\delta P_i)
=\gamma^2
\left[
\operatorname{Var}(r_i)\operatorname{Var}(Z_{i-1})
+\operatorname{Cov}(r_i,Z_{i-1})^2
\right]
$$

Substitution yields

$$
\operatorname{Var}(\delta P_i)
=\gamma^2\omega^4q^2(1-a)^2
\frac{
a^2q+a^2-6aq-2a+q+5
}{
(1+q)(1-aq)^2
}
$$

The corresponding local one-period gross risk-adjusted PnL is

$$
\mathcal S_{\mathrm{gross}}(q)
=\frac{E[\delta P_i]}{\sqrt{\operatorname{Var}(\delta P_i)}}
$$

After cancellation of the scale terms $\gamma$, $\omega^2$, and $q$, this becomes

$$
\mathcal S_{\mathrm{gross}}(q)
=(1-a)
\sqrt{
\frac{1+q}{
a^2q+a^2-6aq-2a+q+5
}
}
$$

This is a local measure. Cumulative PnL has serial dependence, so full-horizon risk-adjusted performance requires the quadratic-form treatment in Part 6.


In [ ]:
#| code-fold: true
#| code-summary: "Show plotting code"
#| label: fig-local-sharpe-vs-q
#| fig-cap: 'Local one-period gross risk-adjusted PnL as a function of $q$. This ratio is scale invariant, but it is not a full-horizon Sharpe ratio because cumulative PnL is serially dependent.'
q_grid = np.linspace(0.001, 0.999, 400)
a_values = [0.60, 0.80, 0.95, 0.99]
omega2 = 1.0

fig, ax = plt.subplots(figsize=(10, 5))
for a in a_values:
    local_sharpe = stationary_formulas(a, q_grid, omega2)["local_sharpe"]
    ax.plot(q_grid, local_sharpe, lw=2, label=f"$a={a}$")
ax.set_xlabel("EWM persistence $q$")
ax.set_ylabel("$\\mathcal{S}_{\\mathrm{gross}}(q)$")
ax.set_title("Local one-period gross risk-adjusted PnL")
ax.legend()
fig.tight_layout()


# Turnover Preview

Transaction costs enter through position changes. For a linear cost rate $c$, the one-period cost is

$$
\mathcal C_i=c|s_i-s_{i-1}|
$$

Since $s_i=-\gamma Z_{i-1}$,

$$
\mathcal C_i=c\gamma|Z_{i-1}-Z_{i-2}|
$$

The difference $\Delta Z_i=Z_i-Z_{i-1}$ is Gaussian with mean zero. Therefore,

$$
E[\mathcal C_i]
=c\gamma\sqrt{\frac{2}{\pi}}\sqrt{\operatorname{Var}(\Delta Z_i)}
$$

Using the EWM recursion, one obtains

$$
\operatorname{Var}(\Delta Z_i)
=
\frac{
2q^2\omega^2(1-a)(3-a-q-aq)
}{
(1+q)(1-aq)
}
$$

This preview is enough to show why the EWM timescale cannot be chosen from gross PnL alone. The same parameter $q$ changes the expected gross edge, the variability of the position, and the expected transaction cost. Parts 8 and 9 will turn this into a net-performance and optimal-half-life problem.


# Formula Summary

| Quantity | Formula |
|:---|:---|
| $\operatorname{Var}(r_i)$ | $2\omega^2(1-a)$ |
| $\operatorname{Cov}(r_i,r_{i-h})$ | $-\omega^2(1-a)^2a^{h-1}$ |
| $\operatorname{Var}(Z_i)$ | $\frac{2\omega^2q^2(1-a)}{(1+q)(1-aq)}$ |
| $\operatorname{Cov}(r_i,Z_{i-1})$ | $-\frac{\omega^2q(1-a)^2}{1-aq}$ |
| $E[\delta P_i]$ | $\gamma\frac{q\omega^2(1-a)^2}{1-aq}$ |
| $\operatorname{Var}(\delta P_i)$ | $\gamma^2\omega^4q^2(1-a)^2\frac{a^2q+a^2-6aq-2a+q+5}{(1+q)(1-aq)^2}$ |
| $\operatorname{Var}(\Delta Z_i)$ | $\frac{2q^2\omega^2(1-a)(3-a-q-aq)}{(1+q)(1-aq)}$ |

: Key stationary formulas for the discrete OU EWM strategy. {#tbl-stationary-formulas}


# Discussion

The main result of this post is the linear-filter representation of the EWM mean-reversion signal. Once the OU process is sampled at a fixed rebalancing interval, returns have an explicit negative autocovariance. The EWM fair-value estimate then transforms the estimated mispricing into an exponentially weighted sum of past returns, and the no-lookahead trading rule turns that filtered history into the position for the next return.

This structure yields closed-form one-period gross PnL formulas. The expected gross PnL is positive because the signal is negatively correlated with the next return, while the one-period variance follows from the product of two centered Gaussian variables. The same state recursion also gives a turnover preview, showing that the EWM persistence $q$ affects both edge and cost.

The remaining limitation is horizon dependence. One-period formulas are useful local diagnostics, but cumulative PnL is serially dependent because both returns and signals share the same history. The next step is therefore to write cumulative PnL as

$$
P_{t,t_0}=\frac{1}{2}r^\top M_q r
$$

where $r$ is the Gaussian return vector and $M_q$ is determined by the EWM memory matrix and the trading window. That quadratic-form representation is the mathematical core of Part 6.
